# Baselines de Churn: DummyClassifier vs Regressão Logística

## Objetivo
Treinar e comparar dois modelos base para previsão de churn **usando o dataset tratado e codificado pelo EDA** (`Telco_customer_churn_ready.csv`):
- **DummyClassifier** (baseline simples)
- **Regressão Logística** (baseline forte e interpretável)

A ideia é validar se um modelo real aprende sinal útil além de um chute ingênuo.

---

## Por que começar por baselines?
Antes de treinar modelos mais complexos (como MLP em PyTorch), precisamos de uma referência mínima de desempenho.
Se um modelo mais sofisticado não supera um baseline simples, ele pode estar adicionando complexidade sem gerar valor real.

## O que será feito (passo a passo) e por quê
1. Carregar o arquivo de saída do EDA (`Telco_customer_churn_ready.csv`).
   - Por quê: esse arquivo já passou por limpeza e encoding, reduzindo risco de inconsistência.
2. Separar variáveis de entrada (`X`) e variável alvo (`y`).
   - Por quê: o modelo precisa aprender a prever `Churn Value` a partir das demais colunas.
3. Dividir treino e teste com estratificação.
   - Por quê: manter a proporção de churn nas duas bases e evitar avaliação enviesada.
4. Treinar DummyClassifier e Regressão Logística no mesmo split.
   - Por quê: comparar modelos em condições justas.
5. Calcular métricas de classificação.
   - Por quê: entender desempenho por diferentes perspectivas (acurácia, recall, F1, ROC-AUC).
6. Registrar experimentos no MLflow, incluindo versão do dataset.
   - Por quê: garantir rastreabilidade, reprodutibilidade e histórico de decisões.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

# Semente para reprodutibilidade: garante o mesmo split e comportamento em reexecuções.
SEED = 42
np.random.seed(SEED)

READY_PATH_CANDIDATES = [
    Path("../data/Telco_customer_churn_ready.csv"),
    Path("data/Telco_customer_churn_ready.csv"),
]

DATA_PATH = next((p for p in READY_PATH_CANDIDATES if p.exists()), None)

assert DATA_PATH is not None, (
    "Arquivo tratado do EDA não encontrado. Rode no notebook de EDA as etapas de Data Readiness e Encoding "
    "para gerar Telco_customer_churn_ready.csv."
)
print(f"Arquivo de entrada encontrado: {DATA_PATH.resolve()}")

Arquivo de entrada encontrado: D:\Projetos\FIAP\Tech Challenge 01\9mlet-tech-challenge-1-churn-prevision\data\Telco_customer_churn_ready.csv


In [2]:
# Carrega o dataset já tratado no EDA.
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print("\nPrimeiras colunas:")
print(df.columns.tolist()[:20])

display(df.head(3))

Shape: (7043, 1163)

Primeiras colunas:
['Zip Code', 'Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV', 'City_Acton', 'City_Adelanto', 'City_Adin', 'City_Agoura Hills', 'City_Aguanga', 'City_Ahwahnee', 'City_Alameda', 'City_Alamo', 'City_Albany', 'City_Albion', 'City_Alderpoint', 'City_Alhambra', 'City_Aliso Viejo']


,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,CLTV,City_Acton,City_Adelanto,City_Adin,...,Streaming TV_Yes,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_One year,Contract_Two year,Paperless Billing_Yes,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check,Churn Value
0,90003,33.964131,-118.272783,2,53.85,108.15,3239,False,False,False,...,False,False,False,False,False,True,False,False,True,1
1,90005,34.059281,-118.307420,2,70.70,151.65,2701,False,False,False,...,False,False,False,False,False,True,False,True,False,1
2,90006,34.048013,-118.293953,8,99.65,820.50,5372,False,False,False,...,True,False,True,False,False,True,False,True,False,1


## Definição de target e features

Como estamos usando o arquivo pronto do EDA, as variáveis categóricas já foram convertidas para numéricas.

- Target: `Churn Value`
- Features: todas as demais colunas

Esse desenho é importante porque evita vazamento de informação e mantém o fluxo alinhado ao problema de negócio: prever churn antes que ele aconteça.

In [3]:
TARGET = "Churn Value"

assert TARGET in df.columns, f"Coluna alvo não encontrada: {TARGET}"

# X contém as variáveis explicativas e y contém a variável que queremos prever.
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

print(f"Target: {TARGET}")
print(f"Total de features: {X.shape[1]}")
print("\nDistribuição do target (proporção):")
print(y.value_counts(normalize=True).rename({0: 'Não churn', 1: 'Churn'}))

Target: Churn Value
Total de features: 1162

Distribuição do target (proporção):
Churn Value
Não churn    0.73463
Churn        0.26537
Name: proportion, dtype: float64


In [4]:
# Separa dados para treinar e para avaliar generalização.
# stratify=y mantém a proporção de churn em treino e teste.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")
print("Proporção de churn em treino e teste:")
print(f"Treino: {y_train.mean():.4f}")
print(f"Teste:  {y_test.mean():.4f}")

Treino: (5634, 1162), Teste: (1409, 1162)
Proporção de churn em treino e teste:
Treino: 0.2654
Teste:  0.2654


In [5]:
# Baseline 1: modelo ingênuo que prevê apenas baseado na classe majoritária.
dummy_model = DummyClassifier(strategy="prior", random_state=SEED)

# Baseline 2: modelo linear forte para classificação tabular.
logreg_model = LogisticRegression(max_iter=5000, random_state=SEED, n_jobs=-1)

dummy_model.fit(X_train, y_train)
logreg_model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,5000
,multi_class,'deprecated'


In [6]:
def evaluate_model(name, model, X_eval, y_eval):
    # Predição de classe para métricas baseadas em rótulo.
    y_pred = model.predict(X_eval)

    # Score/probabilidade para ROC-AUC.
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_eval)[:, 1]
    else:
        y_score = y_pred

    return {
        "modelo": name,
        "accuracy": accuracy_score(y_eval, y_pred),
        "precision": precision_score(y_eval, y_pred, zero_division=0),
        "recall": recall_score(y_eval, y_pred, zero_division=0),
        "f1": f1_score(y_eval, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, y_score),
    }

results = [
    evaluate_model("DummyClassifier(prior)", dummy_model, X_test, y_test),
    evaluate_model("LogisticRegression", logreg_model, X_test, y_test),
]

results_df = pd.DataFrame(results).sort_values(by="roc_auc", ascending=False)
display(results_df.style.format({c: '{:.4f}' for c in ['accuracy','precision','recall','f1','roc_auc']}))

,modelo,accuracy,precision,recall,f1,roc_auc
1,LogisticRegression,0.8048,0.6460,0.5856,0.6143,0.8481
0,DummyClassifier(prior),0.7346,0.0000,0.0000,0.0000,0.5000


## Registro de experimentos no MLflow (parâmetros, métricas e versão do dataset)

Nesta etapa, vamos registrar cada modelo como um experimento no MLflow para garantir rastreabilidade.

O que será registrado em cada run:
- Parâmetros do modelo (ex.: estratégia do Dummy, `max_iter` da Regressão).
- Métricas de avaliação (accuracy, precision, recall, F1 e ROC-AUC).
- Versão do dataset, usando hash SHA-256 do arquivo de entrada.
- Metadados de dados (número de linhas, número de colunas e nome do arquivo).

Isso permite reproduzir resultados e comparar modelos com governança de experimento.

In [8]:
from pathlib import Path
import hashlib
from mlflow.tracking import MlflowClient

# Usa tracking DB local com caminho absoluto para evitar ambiguidades de diretório.
mlflow_db_path = Path("mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{mlflow_db_path.as_posix()}")
print(f"Tracking URI ativo: {mlflow.get_tracking_uri()}")

def file_sha256(path: Path) -> str:
    hash_obj = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hash_obj.update(chunk)
    return hash_obj.hexdigest()

dataset_version = file_sha256(DATA_PATH)
dataset_name = DATA_PATH.name

# Garante experimento com artifact_location local e gravável no ambiente atual.
base_experiment_name = "churn-baselines-etapa1"
local_experiment_name = f"{base_experiment_name}-local"
local_artifacts_dir = Path("mlruns").resolve()
local_artifacts_dir.mkdir(parents=True, exist_ok=True)

client = MlflowClient()
existing_base_exp = client.get_experiment_by_name(base_experiment_name)

if existing_base_exp and "azvef" not in (existing_base_exp.artifact_location or "").lower():
    experiment_name = base_experiment_name
else:
    if client.get_experiment_by_name(local_experiment_name) is None:
        client.create_experiment(
            name=local_experiment_name,
            artifact_location=local_artifacts_dir.as_uri(),
        )
    experiment_name = local_experiment_name

mlflow.set_experiment(experiment_name)
print(f"Experimento ativo: {experiment_name}")

def log_run_to_mlflow(model_name: str, model, metric_row: pd.Series, extra_params: dict):
    with mlflow.start_run(run_name=model_name):
        mlflow.log_param("seed", SEED)
        mlflow.log_param("dataset_name", dataset_name)
        mlflow.log_param("dataset_version_sha256", dataset_version)
        mlflow.log_param("n_linhas", int(df.shape[0]))
        mlflow.log_param("n_colunas", int(df.shape[1]))

        for key, value in extra_params.items():
            mlflow.log_param(key, value)

        mlflow.log_metrics({
            "accuracy": float(metric_row["accuracy"]),
            "precision": float(metric_row["precision"]),
            "recall": float(metric_row["recall"]),
            "f1": float(metric_row["f1"]),
            "roc_auc": float(metric_row["roc_auc"]),
        })

        # Salva o modelo treinado como artefato do experimento.
        mlflow.sklearn.log_model(model, name="model")

        # Registra o dataset usado como artefato para auditoria.
        mlflow.log_artifact(str(DATA_PATH), artifact_path="dataset")

dummy_row = results_df.loc[results_df["modelo"] == "DummyClassifier(prior)"].iloc[0]
logreg_row = results_df.loc[results_df["modelo"] == "LogisticRegression"].iloc[0]

log_run_to_mlflow(
    model_name="DummyClassifier(prior)",
    model=dummy_model,
    metric_row=dummy_row,
    extra_params={"strategy": "prior"},
)

log_run_to_mlflow(
    model_name="LogisticRegression",
    model=logreg_model,
    metric_row=logreg_row,
    extra_params={
        "max_iter": 5000,
        "solver": "lbfgs",
    },
)

print("Experimentos registrados no MLflow com sucesso.")
print(f"Versão do dataset (SHA-256): {dataset_version}")

Tracking URI ativo: sqlite:///D:/Projetos/FIAP/Tech Challenge 01/9mlet-tech-challenge-1-churn-prevision/notebooks/mlflow.db
Experimento ativo: churn-baselines-etapa1-local


2026/04/28 09:32:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/28 09:32:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Experimentos registrados no MLflow com sucesso.
Versão do dataset (SHA-256): 58cf4c8f9c096395ed960d954600b73eb150c822f915d59ac75abb60ae756130


## Como interpretar o resultado
- O **DummyClassifier** funciona como régua mínima. Se um modelo não supera esse baseline, ele não agrega valor prático.
- A **Regressão Logística** costuma ser um baseline forte para dados tabulares e normalmente melhora ROC-AUC e F1.
- Em churn, **recall** da classe positiva também é crítico: perder clientes reais (falso negativo) pode ter alto custo de negócio.